# 05 — Pipeline RAG paramétrable

**Objectif** : Construire un pipeline RAG de bout en bout, paramétrable, pour itérer sur les hyperparamètres et trouver la configuration optimale.

**Paramètres explorables** :
- Taille et overlap des chunks
- Nombre de documents retrieval (`k`)
- Modèle de chat et température
- System prompt

**Pipeline** : Données nettoyées → Chunking → Embedding → FAISS → Retrieval → Génération

---

## 0. Paramètres du pipeline

Tous les hyperparamètres sont regroupés ici. Pour itérer, il suffit de modifier cette cellule et de relancer le notebook.

In [ ]:
# --- Prompt (partagé entre notebooks, voir src/notebooks/prompts.py) ---
from notebooks.prompts import SYSTEM_PROMPT

# --- Chunking ---
CHUNK_MAX_TOKENS = 256
CHUNK_OVERLAP_TOKENS = 20

# --- Retrieval ---
RETRIEVAL_K = 5

# --- Embedding ---
EMBED_MODEL = "mistral-embed"

# --- Génération ---
CHAT_MODEL = "mistral-small-latest"
TEMPERATURE = 0.3

print("=== Paramètres du pipeline ===")
print(f"Chunking       : max={CHUNK_MAX_TOKENS} tokens, overlap={CHUNK_OVERLAP_TOKENS}")
print(f"Retrieval      : k={RETRIEVAL_K}")
print(f"Embed model    : {EMBED_MODEL}")
print(f"Chat model     : {CHAT_MODEL} (T={TEMPERATURE})")
print(f"System prompt  : {len(SYSTEM_PROMPT)} chars")

## 1. Imports & chargement des données nettoyées

In [9]:
from pathlib import Path

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
import mistral_common
from mistral_common.tokens.tokenizers.tekken import Tekkenizer
import pandas as pd

from notebooks.config import DATA_PROCESSED, MISTRAL_API_KEY
from notebooks.mistral import MistralEmbeddings, mistral_chat

df = pd.read_parquet(sorted(DATA_PROCESSED.glob("*_clean.parquet"))[-1])
df_paris = df[df["location_city"].fillna("").str.lower().str.strip() == "paris"].copy().head(200)

print(f"Dataset complet : {len(df)} | Paris (POC) : {len(df_paris)}")

Dataset complet : 9840 | Paris (POC) : 200


## 2. Chunking paramétrable

In [10]:
data_dir = Path(mistral_common.__file__).parent / "data"
tokenizer = Tekkenizer.from_file(str(sorted(data_dir.glob("tekken_*.json"), reverse=True)[0]))


def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, bos=False, eos=False))


def _has_value(val):
    if val is None:
        return False
    try:
        if pd.isna(val):
            return False
    except (ValueError, TypeError):
        pass
    return not (isinstance(val, str) and val.strip() == "")


def build_parent_text(row):
    parts = [f"# {row['title_fr']}"]
    parts.append(f"\n**Date** : {row['date_display']}")
    parts.append(f"**Lieu** : {row['location_name']}, {row['location_city']}")
    if row.get("location_address"):
        parts.append(f"**Adresse** : {row['location_address']}")
    if _has_value(row.get("keywords_fr")):
        kw = row["keywords_fr"]
        if isinstance(kw, list):
            kw = ", ".join(kw)
        parts.append(f"**Mots-clés** : {kw}")
    if _has_value(row.get("conditions_fr")):
        parts.append(f"**Conditions** : {row['conditions_fr']}")
    parts.append(f"\n{row['description_clean']}")
    return "\n".join(parts)


def build_chunk_header(row):
    header = f"Titre : {row['title_fr']}"
    if _has_value(row.get("keywords_fr")):
        kw = row["keywords_fr"]
        if isinstance(kw, list):
            kw = ", ".join(kw)
        header += f"\nMots-clés : {kw}"
    return header


splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_MAX_TOKENS,
    chunk_overlap=CHUNK_OVERLAP_TOKENS,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=count_tokens,
)

chunks_data: list[dict] = []
parent_data: dict[str, str] = {}

for _, row in df_paris.iterrows():
    uid = str(row["uid"])
    parent_data[uid] = build_parent_text(row)

    header = build_chunk_header(row)
    description = (row["description_clean"] or "").strip() or row["title_fr"]
    fragments = splitter.split_text(description)

    for i, fragment in enumerate(fragments):
        chunk_text = f"{header}\n\n{fragment}"
        chunks_data.append({"uid": uid, "chunk_id": i, "text": chunk_text})

print(f"Chunks : {len(chunks_data)} | Parents : {len(parent_data)}")

Chunks : 350 | Parents : 200


## 3. Embedding + indexation FAISS

In [11]:
embedder = MistralEmbeddings(api_key=MISTRAL_API_KEY, model=EMBED_MODEL)

texts = [c["text"] for c in chunks_data]
metadatas = [{"uid": c["uid"], "chunk_id": str(c["chunk_id"])} for c in chunks_data]

print(f"Embedding {len(texts)} chunks via {EMBED_MODEL}...")
embeddings = embedder.embed_documents(texts)

vectorstore = FAISS.from_embeddings(
    text_embeddings=list(zip(texts, embeddings)),
    embedding=embedder,
    metadatas=metadatas,
)
print(f"Index FAISS : {vectorstore.index.ntotal} vecteurs")

Embedding 350 chunks via mistral-embed...
Index FAISS : 350 vecteurs


## 4. Pipeline RAG — fonction `ask()`

In [12]:
def ask(question: str, k: int = RETRIEVAL_K) -> dict:
    results = vectorstore.similarity_search(question, k=k)

    seen_uids: set[str] = set()
    sources: list[dict] = []
    for doc in results:
        uid = doc.metadata.get("uid")
        if uid and uid not in seen_uids:
            seen_uids.add(uid)
            parent = parent_data.get(uid)
            if parent:
                sources.append({"uid": uid, "text": parent})

    if not sources:
        return {"question": question, "answer": "Aucun événement pertinent trouvé.", "sources": []}

    context = "\n---\n".join(f"[{i+1}] {s['text']}" for i, s in enumerate(sources))
    user_prompt = f"Contexte (extraits d'événements) :\n{context}\n\nQuestion de l'utilisateur :\n{question}"

    answer = mistral_chat(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        api_key=MISTRAL_API_KEY,
        model=CHAT_MODEL,
        temperature=TEMPERATURE,
    )

    return {"question": question, "answer": answer, "sources": sources}


print("Fonction ask() prête")

Fonction ask() prête


## 5. Tests de la chaîne RAG

In [13]:
TEST_QUERIES = [
    "Où et à quelles dates se déroule la visite-atelier 'Prendre la mesure du temps' ?",
    "Faut-il payer ou s'inscrire pour participer aux 'Nuits des étoiles au parc Montsouris' ?",
    "Quel équipement est fourni aux auditeurs lors de l'expérience sonore 'Les Sons de Voyager' ?",
    "Qui animera la scène TWITCH lors du Festival MVP - Ma Vie Pro ?",
    "Pourquoi l'accès aux tribunes n'est-il pas permis lors de la visite de l'Église Saint-Vincent-de-Paul ?",
]

for question in TEST_QUERIES:
    print(f"\n{'=' * 80}")
    print(f"QUESTION : {question}")
    print(f"{'=' * 80}")

    result = ask(question)
    print(f"\n{result['answer']}")
    print(f"\nSources ({len(result['sources'])}) :")
    for s in result["sources"]:
        print(f"  - [{s['uid']}] {s['text'].split(chr(10))[0]}")


QUESTION : Où et à quelles dates se déroule la visite-atelier 'Prendre la mesure du temps' ?

La visite-atelier **"Prendre la mesure du temps"** se déroule au **Musée de la poste à Paris (34 Boulevard de Vaugirard, 75015)**.

Les dates sont **du 20 septembre 2025 à 12h30 au 21 septembre 2025 à 12h30**.

Sources (5) :
  - [6670838] # Visite-atelier : Prendre la mesure du temps
  - [43521113] # Visite libre du Mur des je t'aime en présence des artistes
  - [65315210] # Visite du site historique du Collège de France
  - [40948393] # Exposition des travaux des élèves des lycées professionnels et des écoles d’art de l’académie de Paris
  - [11372405] # Une bibliothèque d'art dans un monument historique

QUESTION : Faut-il payer ou s'inscrire pour participer aux 'Nuits des étoiles au parc Montsouris' ?

Non, il n'est pas nécessaire de payer ni de s'inscrire pour participer aux **Nuits des étoiles au parc Montsouris**. L'événement est gratuit et ouvert à tous sans réservation.

Sources (2) :

## 6. Comparaison avec/sans RAG

In [14]:
question_test = "Pourquoi l'accès aux tribunes n'est-il pas permis lors de la visite de l'Église Saint-Vincent-de-Paul ?"

print("=== SANS RAG (LLM seul) ===")
response_no_rag = mistral_chat(
    messages=[{"role": "user", "content": question_test}],
    api_key=MISTRAL_API_KEY,
    model=CHAT_MODEL,
    temperature=TEMPERATURE,
)
print(response_no_rag[:500] + "..." if len(response_no_rag) > 500 else response_no_rag)

print(f"\n{'=' * 80}")

print("\n=== AVEC RAG ===")
result = ask(question_test)
print(result["answer"])

=== SANS RAG (LLM seul) ===
L'accès aux tribunes de l'Église Saint-Vincent-de-Paul à Paris n'est pas permis lors des visites pour des raisons de **sécurité et de préservation du monument**.

1. **Sécurité des visiteurs** : Les tribunes, situées en hauteur, peuvent présenter des risques de chute ou d'effondrement, surtout si elles ne sont pas conçues pour un accès public régulier.
2. **Protection du patrimoine** : L'église, classée aux **Monuments Historiques**, nécessite une conservation rigoureuse. Une fréquentation non c...


=== AVEC RAG ===
L'accès aux tribunes de l'Église Saint-Vincent-de-Paul n'est pas permis en raison de la **restauration des peintures à la cire de Flandrin**, lancée par la Mairie de Paris. Seule la terrasse reste exceptionnellement accessible lors de cette visite.


## Bilan

Pipeline RAG fonctionnel et paramétrable :

| Paramètre | Valeur |
|---|---|
| Chunk max tokens | `CHUNK_MAX_TOKENS` |
| Chunk overlap | `CHUNK_OVERLAP_TOKENS` |
| Retrieval k | `RETRIEVAL_K` |
| Modèle embedding | `EMBED_MODEL` |
| Modèle chat | `CHAT_MODEL` |
| Température | `TEMPERATURE` |
| System prompt | Personnalisable |

Pour itérer : modifier les paramètres en section 0 et relancer le notebook.

---
**Suite** : `06.0-kl-evaluation.ipynb` — évaluation RAGAS